# 1.4 Feature engineering

(1) Load preprocessed train, (2) fit feature engineering, (3) save CSV and pickle.

New columns are allowed here: `house_age`, ordinal `condition`, one-hot `location`. Numeric features are scaled; `price` is not.

| Artifact | Path |
|---|---|
| Input | `1-experimentation/data/data_preprocessed_train.csv` |
| Output Featured Train | `1-experimentation/data/data_featured_train.csv` |
| Output Feature Engineer | `1-experimentation/models/feature_engineer.pkl` |


In [1]:
%pip install -q pandas scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn import set_config
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

set_config(transform_output="pandas")

TARGET = "price"
REFERENCE_YEAR = 2026
CONDITION_ORDER = ["Poor", "Fair", "Good", "Excellent"]
FEATURE_NUMERIC = ["sqft", "bedrooms", "bathrooms", "house_age"]
FEATURE_ORDINAL = ["condition"]
FEATURE_NOMINAL = ["location"]

PREPROCESSED_TRAIN_PATH = "../data/data_preprocessed_train.csv"
FEATURED_TRAIN_PATH = "../data/data_featured_train.csv"
FEATURE_ENGINEER_PATH = "../models/feature_engineer.pkl"

Path("../models").mkdir(parents=True, exist_ok=True)


## 1. Load preprocessed train

In [3]:
train_df = pd.read_csv(PREPROCESSED_TRAIN_PATH)

print(f"Rows: {train_df.shape[0]} | Columns: {train_df.shape[1]}")
print(f"Columns: {list(train_df.columns)}")
train_df.head()


Rows: 67 | Columns: 7
Columns: ['price', 'sqft', 'bedrooms', 'bathrooms', 'location', 'year_built', 'condition']


,price,sqft,bedrooms,bathrooms,location,year_built,condition
0,495000.0,1950.0,3.0,2.0,Urban,1981.0,Good
1,320000.0,1700.0,2.0,1.5,Rural,1961.0,Fair
2,615000.0,2230.0,3.0,2.0,Downtown,1986.0,Good
3,398000.0,1680.0,2.0,2.0,Suburb,1968.0,Fair
4,357000.0,1580.0,2.0,1.5,Suburb,1960.0,Fair


## 2. Fit feature engineering

The pickled pipeline includes:

1. Add `house_age` (`reference_year - year_built`) and drop `year_built`
2. Scale numeric columns (`sqft`, `bedrooms`, `bathrooms`, `house_age`)
3. Encode `condition` as Poor < Fair < Good < Excellent
4. One-hot encode `location`

`price` is passed through unchanged.


In [4]:
class AddHouseAge(BaseEstimator, TransformerMixin):
    def __init__(self, reference_year=2026, source="year_built"):
        self.reference_year = reference_year
        self.source = source

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        out = X.copy()
        out["house_age"] = self.reference_year - out[self.source]
        return out.drop(columns=[self.source])


feature_engineer = Pipeline(
    [
        ("add_house_age", AddHouseAge(reference_year=REFERENCE_YEAR)),
        (
            "encode",
            ColumnTransformer(
                transformers=[
                    ("num", StandardScaler(), FEATURE_NUMERIC),
                    (
                        "ord",
                        OrdinalEncoder(
                            categories=[CONDITION_ORDER],
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                        FEATURE_ORDINAL,
                    ),
                    (
                        "nom",
                        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        FEATURE_NOMINAL,
                    ),
                ],
                remainder="passthrough",
                verbose_feature_names_out=False,
            ),
        ),
    ]
)

feature_engineer.fit(train_df)
train_featured = feature_engineer.transform(train_df)

print(f"Rows: {train_featured.shape[0]} | Columns: {train_featured.shape[1]}")
print(f"Columns: {list(train_featured.columns)}")
train_featured.head()


Rows: 67 | Columns: 12
Columns: ['sqft', 'bedrooms', 'bathrooms', 'house_age', 'condition', 'location_Downtown', 'location_Mountain', 'location_Rural', 'location_Suburb', 'location_Urban', 'location_Waterfront', 'price']


,sqft,bedrooms,bathrooms,house_age,condition,location_Downtown,location_Mountain,location_Rural,location_Suburb,location_Urban,location_Waterfront,price
0,-0.346309,0.226617,-0.201016,0.025525,2.0,0.0,0.0,0.0,0.0,1.0,0.0,495000.0
1,-0.741019,-1.038661,-0.813201,1.062004,1.0,0.0,0.0,1.0,0.0,0.0,0.0,320000.0
2,0.095767,0.226617,-0.201016,-0.233594,2.0,1.0,0.0,0.0,0.0,0.0,0.0,615000.0
3,-0.772596,-1.038661,-0.201016,0.699236,1.0,0.0,0.0,0.0,1.0,0.0,0.0,398000.0
4,-0.930481,-1.038661,-0.813201,1.113828,1.0,0.0,0.0,0.0,1.0,0.0,0.0,357000.0


## 3. Save artifacts

In [5]:
train_featured.to_csv(FEATURED_TRAIN_PATH, index=False)
with open(FEATURE_ENGINEER_PATH, "wb") as file:
    pickle.dump(feature_engineer, file)

print(f"Wrote {FEATURED_TRAIN_PATH}")
print(f"Wrote {FEATURE_ENGINEER_PATH}")


Wrote ../data/data_featured_train.csv
Wrote ../models/feature_engineer.pkl
